In [8]:
import os
import json
import glob
import re
import numpy as np
import xarray as xr
from typing import Dict, Tuple, List, Optional
from joblib import Parallel, delayed
from cftime import num2date, DatetimeNoLeap
from datetime import timedelta
import matplotlib.pyplot as plt
from scipy.stats import t as t_dist
try:
    import xskillscore as xs
except Exception:
    xs = None
try:
    from scipy.stats import t as t_dist
except Exception:
    t_dist = None

In [12]:
class BiasSigAtLevelsCalculator:
    """
    Compute seasonal bias maps + significance for:
      (a) 3D pressure-level fields → dims: (season, plev, lat, lon)
      (b) 2D surface fields        → dims: (season, lat, lon)

    For each (var, exp) writes a NetCDF:
      - bias_mean
      - pvalue
      - sig_mask                  [p < alpha]
      - area_mean_bias           (per plev for 3D; scalar per season for 2D)

    Assumptions & dependencies are the same as before.
    """

    def __init__(
        self,
        regnam: str,
        tstart: str,
        tend: str,
        path_in: str,
        out_path: str,
        model_list: List[str],
        ref_dict: Dict[str, Dict],
        exp_dict: Dict[str, Dict],
        var_dict: Dict[str, Dict],
        seasons: Optional[Dict[str, List[int]]] = None,
        force: bool = False,
    ):
        self.regnam = regnam
        self.tstart = tstart
        self.tend = tend
        self.path_in = path_in
        self.out_path = out_path
        self.model_list = model_list
        self.ref_dict = ref_dict
        self.exp_dict = exp_dict
        self.var_dict = var_dict
        self.force = force

        self.seasons = seasons or {
            "DJF": [12, 1, 2],
            "MAM": [3, 4, 5],
            "JJA": [6, 7, 8],
            "SON": [9, 10, 11],
            "ANN": list(range(1, 13)),
        }

        os.makedirs(self.out_path, exist_ok=True)
        self.model_data_cache: Dict[str, xr.Dataset] = {}

    # ------------------------- Public API -------------------------

    def compute(
        self,
        variables: List[str],
        levels_hpa: Tuple[float, ...] = (200., 500., 850.),
        alpha: float = 0.05,
        tol_hpa: float = 25.0,
    ):
        """Compute & save seasonal bias + significance; auto-handles surface vs levels."""
        self._load_all_model_data()

        for var in variables:
            if var not in self.var_dict and var not in self.ref_dict:
                print(f"[WARN] {var} missing in var_dict/ref_dict; skipping.")
                continue

            print(f"[INFO] Variable: {var}")
            obs = self._read_reference_data(var)
            period = f"{self.tstart[:4]}-{self.tend[:4]}"

            for exp in self.model_list:
                print(f"[INFO]   Experiment: {exp}")
                mod = self._read_model_data(var, self.model_data_cache[exp])

                # Normalize (lev→plev in hPa if present) and standardize dims
                obs = self._normalize_levels(obs)
                mod = self._normalize_levels(mod)

                has_plev = ("plev" in obs.dims) or ("plev" in mod.dims)

                if not has_plev:
                    # 2D surface path
                    out_file = os.path.join(self.out_path, f"{var}_{self.regnam}_biassig2D_{exp}_{period}.nc")
                    if os.path.exists(out_file) and not self.force:
                        print(f"[SKIP] {out_file} exists.")
                        continue

                    ds_out = self._compute_biassig_dataset_surface(obs, mod, alpha=alpha)
                    reg_lat, reg_lon = self._define_region()
                    ds_out.attrs.update(
                        dict(
                            variable=var,
                            region=self.regnam,
                            region_lat=f"{reg_lat[0]}..{reg_lat[1]}",
                            region_lon=f"{reg_lon[0]}..{reg_lon[1]}",
                            tstart=self.tstart,
                            tend=self.tend,
                            alpha=alpha,
                            levels_requested="(surface)",
                            levels_selected="(surface)",
                        )
                    )
                    ds_out.to_netcdf(out_file)
                    print(f"[DONE] Wrote {out_file}")
                    continue

                # 3D pressure-level path
                out_file = os.path.join(self.out_path, f"{var}_{self.regnam}_biassigPL_{exp}_{period}.nc")
                if os.path.exists(out_file) and not self.force:
                    print(f"[SKIP] {out_file} exists.")
                    continue

                if "plev" not in obs.dims or "plev" not in mod.dims:
                    raise ValueError(f"[ERROR] {var}: one of obs/model lacks 'plev' while the other has it.")

                # Select nearest native levels within tolerance for both, but label with requested targets
                obs_sel, obs_targets = self._select_levels(obs, levels_hpa, tol_hpa)
                mod_sel, mod_targets = self._select_levels(mod, levels_hpa, tol_hpa)

                # Retain only common target labels to ensure matching stacks
                common_targets = [p for p in levels_hpa if (p in obs_targets) and (p in mod_targets)]
                if not common_targets:
                    raise ValueError("No common target levels between obs and model within tolerance.")

                # enforce same order for both
                obs_sel = obs_sel.sel(plev=common_targets)
                mod_sel = mod_sel.sel(plev=common_targets)

                ds_out = self._compute_biassig_dataset_levels(obs_sel, mod_sel, alpha=alpha)
                ds_out = ds_out.assign_coords(plev=("plev", np.array(common_targets, dtype=float)))

                reg_lat, reg_lon = self._define_region()
                ds_out.attrs.update(
                    dict(
                        variable=var,
                        region=self.regnam,
                        region_lat=f"{reg_lat[0]}..{reg_lat[1]}",
                        region_lon=f"{reg_lon[0]}..{reg_lon[1]}",
                        tstart=self.tstart,
                        tend=self.tend,
                        alpha=alpha,
                        levels_requested=",".join(map(str, levels_hpa)),
                        levels_selected=",".join(map(lambda x: f"{x:.1f}", ds_out["plev"].values)),
                    )
                )

                ds_out.to_netcdf(out_file)
                print(f"[DONE] Wrote {out_file}")

    # ------------------------- IO & Prep -------------------------

    def _load_all_model_data(self):
        for exp in self.model_list:
            print(f"[LOAD] Model data: {exp}")
            path = self.path_in.replace("%(CASENAME)", self.exp_dict[exp]["run"])
            files = sorted(glob.glob(os.path.join(path, "*_h0.nc")))
            sy, ey = int(self.tstart[:4]), int(self.tend[:4])

            sel = []
            for f in files:
                base = os.path.basename(f)
                try:
                    yr = int(base.split("_")[0])
                    if sy <= yr <= ey:
                        sel.append(f)
                except Exception:
                    print(f"[WARN] Skipping unexpected file: {base}")

            if not sel:
                raise FileNotFoundError(f"No model files found for {exp} in {path} within {sy}-{ey}")

            ds = xr.open_mfdataset(
                sel, combine="nested", concat_dim="time",
                coords="minimal", compat="override", decode_times=False
            )
            ds = self._fix_coordinates(ds)
            ds = self._assign_cftime_time(ds)
            ds = self._subset_region(ds)
            self.model_data_cache[exp] = ds

    def _read_reference_data(self, var: str) -> xr.DataArray:
        r = self.ref_dict[var]
        # Support both styles: "{year}" and "%(year)"
        template = r["template"]
        glob_pat = template.replace("%(year)", "*").replace("{year}", "*")
        all_files = sorted(glob.glob(os.path.join(r["run"], glob_pat)))
        sy, ey = int(self.tstart[:4]), int(self.tend[:4])
    
        sel = []
        for f in all_files:
            base = os.path.basename(f)
            # Robust year extraction: first 4-digit year in filename
            import re
            m = re.search(r"(?<!\d)(19|20)\d{2}(?!\d)", base)
            if not m:
                print(f"[WARN] Skipping unexpected ref file (no YYYY found): {base}")
                continue
            yr = int(m.group(0))
            if sy <= yr <= ey:
                sel.append(f)
    
        if not sel:
            raise FileNotFoundError(
                f"No reference files found for {var} under {r['run']} {r['template']} "
                f"(after glob '{glob_pat}') in {sy}-{ey}"
            )
        ds = xr.open_mfdataset(
            sel, combine="nested", concat_dim="time",
            coords="minimal", compat="override", engine="netcdf4", decode_times=False
        )
        ds = self._fix_coordinates(ds)
        ds = self._assign_cftime_time(ds)
        ds = self._subset_region(ds)

        alias = r["alias"]
        fscl = r["fscl"]
        if var in ds:
            da = ds[var]
        elif alias in ds:
            da = ds[alias]
        else:
            if var == "PRECT" and "PRECC" in ds and "PRECL" in ds:
                da = ds["PRECC"] + ds["PRECL"]
            elif var == "PRECST" and "PRECSC" in ds and "PRECSL" in ds:
                da = ds["PRECSC"] + ds["PRECSL"]
            else:
                raise KeyError(f"Reference variable not found: {var} or alias {alias}")
        da = da * fscl
        da = self._normalize_levels(da)
        da = self._order_dims(da)
        return da

    def _read_model_data(self, var: str, ds_model: xr.Dataset) -> xr.DataArray:
        vinfo = self.var_dict.get(var, {})
        alias = vinfo.get("alias", var)
        fscl = vinfo.get("fscl", 1.0)

        ds = ds_model
        if var in ds:
            da = ds[var]
        elif alias in ds:
            da = ds[alias]
        else:
            if var == "PRECT" and ("PRECC" in ds) and ("PRECL" in ds):
                da = ds["PRECC"] + ds["PRECL"]
            elif var == "PRECST" and ("PRECSC" in ds) and ("PRECSL" in ds):
                da = ds["PRECSC"] + ds["PRECSL"]
            elif var == "CRE" and ("LWCF" in ds) and ("SWCF" in ds):
                da = ds["LWCF"] + ds["SWCF"]
            else:
                raise KeyError(f"Model variable not found or derivable: {var} (alias {alias})")
        da = da * fscl
        da = self._normalize_levels(da)
        da = self._order_dims(da)
        return da

    # ------------------------- Core Compute -------------------------

    def _compute_biassig_dataset_levels(self, obs: xr.DataArray, mod: xr.DataArray, alpha: float = 0.05) -> xr.Dataset:
        weights = self._generate_coslat_weight(mod)

        bias_list, pval_list, mask_list, area_list = [], [], [], []
        seasons_order = list(self.seasons.keys())

        for season, months in self.seasons.items():
            if season == "ANN":
                obs_s, mod_s = obs, mod
            else:
                obs_s = obs.where(obs["time.month"].isin(months), drop=True)
                mod_s = mod.where(mod["time.month"].isin(months), drop=True)

            if obs_s.sizes.get("time", 0) == 0 or mod_s.sizes.get("time", 0) == 0:
                tmpl = mod.isel(time=0, drop=True)
                empty = xr.zeros_like(tmpl) * np.nan
                bias_list.append(empty)
                pval_list.append(empty)
                mask_list.append(xr.zeros_like(tmpl, dtype=bool))
                area_list.append(np.full((tmpl.sizes.get("plev", 1),), np.nan))
                continue

            bias = mod_s - obs_s
            bias_mean = bias.mean("time")
            pvals = self._pval_one_sample_zero_with_ar1(bias, dim="time")
            sig_mask = pvals < alpha

            area_mean = (bias_mean * weights).mean(("lat", "lon"), skipna=True).compute()
            if "plev" not in area_mean.dims:
                area_mean = area_mean.expand_dims(plev=[np.nan])

            bias_list.append(bias_mean)
            pval_list.append(pvals)
            mask_list.append(sig_mask)
            area_list.append(area_mean)

        ds_out = xr.Dataset(
            {
                "bias_mean": xr.concat(bias_list, dim="season"),
                "pvalue": xr.concat(pval_list, dim="season"),
                "sig_mask": xr.concat(mask_list, dim="season"),
                "area_mean_bias": xr.concat(area_list, dim="season"),
            },
            coords={"season": seasons_order}
        )
        return ds_out

    def _compute_biassig_dataset_surface(self, obs: xr.DataArray, mod: xr.DataArray, alpha: float = 0.05) -> xr.Dataset:
        """Same metrics as levels version but without 'plev'."""
        weights = self._generate_coslat_weight(mod)

        bias_list, pval_list, mask_list, area_list = [], [], [], []
        seasons_order = list(self.seasons.keys())

        for season, months in self.seasons.items():
            if season == "ANN":
                obs_s, mod_s = obs, mod
            else:
                obs_s = obs.where(obs["time.month"].isin(months), drop=True)
                mod_s = mod.where(mod["time.month"].isin(months), drop=True)

            if obs_s.sizes.get("time", 0) == 0 or mod_s.sizes.get("time", 0) == 0:
                tmpl = mod.isel(time=0, drop=True)
                empty = xr.zeros_like(tmpl) * np.nan
                bias_list.append(empty)
                pval_list.append(empty)
                mask_list.append(xr.zeros_like(tmpl, dtype=bool))
                area_list.append(np.array(np.nan))
                continue

            bias = mod_s - obs_s
            bias_mean = bias.mean("time")
            pvals = self._pval_one_sample_zero_with_ar1(bias, dim="time")
            sig_mask = pvals < alpha

            area_mean = (bias_mean * weights).mean(("lat", "lon"), skipna=True).compute()

            bias_list.append(bias_mean)
            pval_list.append(pvals)
            mask_list.append(sig_mask)
            area_list.append(area_mean)

        ds_out = xr.Dataset(
            {
                "bias_mean": xr.concat(bias_list, dim="season"),
                "pvalue": xr.concat(pval_list, dim="season"),
                "sig_mask": xr.concat(mask_list, dim="season"),
                "area_mean_bias": xr.DataArray(area_list, coords={"season": seasons_order}, dims=("season",)),
            },
            coords={"season": seasons_order}
        )
        return ds_out

    # ------------------------- Math Utils -------------------------

    @staticmethod
    def _lag1_autocorr(arr: np.ndarray, axis: int = 0) -> np.ndarray:
        x = np.moveaxis(arr, axis, 0)
        if x.shape[0] < 2:
            return np.zeros_like(np.nanmean(x, axis=0))
        x0, x1 = x[:-1], x[1:]
        m0 = np.nanmean(x0, axis=0)
        m1 = np.nanmean(x1, axis=0)
        num = np.nanmean((x0 - m0) * (x1 - m1), axis=0)
        den = np.sqrt(np.nanmean((x0 - m0) ** 2, axis=0) * np.nanmean((x1 - m1) ** 2, axis=0))
        r1 = num / den
        return np.clip(r1, -0.99, 0.99)

    def _pval_one_sample_zero_with_ar1(self, da: xr.DataArray, dim: str = "time") -> xr.DataArray:
        if dim not in da.dims:
            raise ValueError(f"Dimension '{dim}' not in DataArray")
        da = da.transpose(dim, ...)
        space = [d for d in da.dims if d != dim]
        stacked = da.stack(z=space)  # (time, z)
        v = stacked.values
        if hasattr(v, "compute"):
            v = v.compute()

        n_t = np.sum(np.isfinite(v), axis=0)
        mask = n_t >= 3

        p = np.full(n_t.shape, np.nan, dtype=float)
        if not np.any(mask):
            p_da = xr.DataArray(p, coords={"z": stacked["z"]}, dims="z").unstack("z")
            return p_da.transpose(*space)

        v_sub = v[:, mask]
        mean_t = np.nanmean(v_sub, axis=0)
        std_t  = np.nanstd(v_sub, axis=0, ddof=1)

        r1_sub = self._lag1_autocorr(v_sub, axis=0)
        neff_sub = n_t[mask] * (1.0 - r1_sub) / (1.0 + r1_sub)
        neff_sub = np.where(np.isfinite(neff_sub) & (neff_sub >= 2.0), neff_sub, 2.0)
        dof_sub  = neff_sub - 1.0

        se = std_t / np.sqrt(neff_sub)
        valid_sub = np.isfinite(se) & (se > 0) & np.isfinite(dof_sub) & (dof_sub > 0)

        tstat = np.full_like(mean_t, np.nan, dtype=float)
        tstat[valid_sub] = mean_t[valid_sub] / se[valid_sub]

        from scipy.stats import t as t_dist
        p_sub = np.full_like(mean_t, np.nan, dtype=float)
        ok = valid_sub
        p_sub[ok] = 2.0 * (1.0 - t_dist.cdf(np.abs(tstat[ok]), df=dof_sub[ok]))

        p[mask] = p_sub

        p_da = xr.DataArray(p, coords={"z": stacked["z"]}, dims="z").unstack("z")
        return p_da.transpose(*space)

    # ------------------------- Coordinates & Levels -------------------------

    @staticmethod
    def _fix_coordinates(ds: xr.Dataset) -> xr.Dataset:
        if "lon" in ds.coords and ds.lon.min() > -1.0:
            ds = ds.assign_coords(lon=((ds.lon + 180) % 360 - 180)).sortby("lon")
        return ds

    def _assign_cftime_time(self, ds: xr.Dataset) -> xr.Dataset:
        dates = xr.cftime_range(start=self.tstart, end=self.tend, freq="MS", calendar="365_day")
        if "time" not in ds.dims:
            raise ValueError("Dataset missing 'time' dimension.")
        if ds.sizes["time"] != len(dates):
            raise ValueError(f"[ERROR] Time mismatch: {ds.sizes['time']} vs expected {len(dates)}")
        ds = ds.assign_coords(time=("time", dates))
        return ds

    def _subset_region(self, ds: xr.Dataset) -> xr.Dataset:
        (lat_rng, lon_rng) = self._define_region()
        ds = ds.sel(lat=slice(*lat_rng), lon=slice(*lon_rng))
        return ds

    def _order_dims(self, da: xr.DataArray) -> xr.DataArray:
        order = [d for d in ("time", "plev", "lat", "lon") if d in da.dims]
        return da.transpose(*order)

    @staticmethod
    def _detect_plev_name(ds_or_da: xr.Dataset) -> Optional[str]:
        for cand in ("plev", "lev", "level", "pressure"):
            if (hasattr(ds_or_da, "dims") and cand in ds_or_da.dims) or \
               (hasattr(ds_or_da, "coords") and cand in ds_or_da.coords):
                return cand
        return None

    def _normalize_levels(self, da: xr.DataArray) -> xr.DataArray:
        pname = self._detect_plev_name(da.to_dataset(name="_tmp"))
        if pname is None:
            return da  # surface variable
        p = da.coords[pname]
        p_numeric = np.asarray(p.values, dtype=float)
        p_hpa = p / 100.0 if np.nanmedian(p_numeric) > 2000 else p
        da = da.assign_coords(plev=p_hpa)
        if "plev" != pname:
            da = da.swap_dims({pname: "plev"}).drop_vars(pname, errors="ignore")
        da = da.sortby("plev", ascending=False)
        return da

    def _select_levels(self, da: xr.DataArray, levels_hpa, tol_hpa: float = 25.0):
        if "plev" not in da.dims:
            raise ValueError("select_levels called on DataArray without 'plev' dimension.")
        avail = da["plev"].values.astype(float)
        idxs, kept_targets = [], []
        for p in levels_hpa:
            i = int(np.nanargmin(np.abs(avail - p)))
            if np.isfinite(avail[i]) and abs(avail[i] - p) <= tol_hpa:
                idxs.append(i)
                kept_targets.append(float(p))
            else:
                print(f"[WARN] level {p} hPa not within ±{tol_hpa} hPa; skipping.")
        if not idxs:
            raise ValueError("None of the requested levels found within tolerance.")
        da_sel = da.isel(plev=xr.DataArray(idxs, dims="plev"))
        da_sel = da_sel.assign_coords(plev=("plev", np.array(kept_targets, dtype=float)))
        da_sel = self._order_dims(da_sel)
        return da_sel, kept_targets

    def _generate_coslat_weight(self, sample: xr.DataArray) -> xr.DataArray:
        wlat = np.cos(np.deg2rad(sample["lat"]))
        ref = sample.isel(time=0, **({} if "plev" not in sample.dims else {"plev": 0}))
        w = wlat.broadcast_like(ref)
        if "plev" in sample.dims:
            w = w.expand_dims(plev=sample["plev"])
            order = [d for d in sample.dims if d != "time"]
            w = w.transpose(*order)
        return w

    # ------------------------- Regions -------------------------

    def _define_region(self) -> Tuple[Tuple[float, float], Tuple[float, float]]:
        reg = {
            "global":   ((-90, 90), (-180, 180)),
            "Atlantic": ((5, 55),   (-95, -40)),
            "CONUS":    ((25, 50),  (-125, -95)),
            "Antarctic":((-90, -50),( -180, 180)),
            "PolarN":   ((50, 90),  (-180, 180)),
            "Greenland":((60, 85),  ( -75, -10)),
        }
        if self.regnam not in reg:
            raise KeyError(f"Unknown region '{self.regnam}'")
        return reg[self.regnam]

In [13]:
if __name__ == "__main__":
    # --- paths ---
    top_path  = "/pscratch/sd/z/zhan391/seacrogs_scratch"
    data_path = f"{top_path}/post_data"
    out_path  = "/pscratch/sd/z/zhan391/SEACROGS_project/paper_material/method_paper/fig_data/horizontal_bias"
    os.makedirs(out_path, exist_ok=True)

    # --- experiments metadata ---
    exp_json = f"{data_path}/scripts/ml_exp_info.json"
    with open(exp_json, "r") as f:
        exp_dict: Dict[str, Dict] = json.load(f)
    for _, v in exp_dict.items():
        v.setdefault("run", v.get("case", v.get("CASENAME", "")))
        v.setdefault("ref", "ERA5")

    # --- time / freq / region ---
    tstart = "2012-01-01"
    tend   = "2016-12-01"    
    regnam = "global"
    freq   = "monthly"
    tol_hpa = 25.0
    plevs  = (200., 500., 850.),  # ignored for surface vars
    alpha  = 0.05  # significance level

    # --- model file root ---
    path_template = f"{data_path}/%(CASENAME)/{freq}"
    
    # --- choose variables & levels ---
    # Use pressure-level vars; surface-only vars are skipped by this routine
    variables = ["U","V","T","Q"]
    levels_hpa = (200., 500., 850.) # change as needed
    alpha = 0.05
    tol_hpa = 25.0                  # nearest-level tolerance in hPa

    # --- variable metadata (units/scales); extend as needed ---
    VAR_DICT = {
        "U": {"alias": "U", "fscl": 1.0},
        "V": {"alias": "V", "fscl": 1.0},
        "T": {"alias": "T", "fscl": 1.0},
        "Q": {"alias": "Q", "fscl": 1.0},
        "Z": {"alias": "Z3", "fscl": 1.0},   # common model alias
    }
    
    # --- reference dataset info (expand per-variable as class expects) ---
    # Your ERA5 files should match this template pattern under ref_root.
    # Adjust 'alias' to the variable name inside the ERA5 files.
    ref_root = f"{data_path}/ERA5"
    ref_template = "monthly/ERA5_analysis_monthly_%(year).nc"
    REF_DICT = {
        var: {
            "source": "ERA5",
            "run": ref_root,
            "template": ref_template,
            "alias": VAR_DICT[var].get("alias", var),
            "fscl": VAR_DICT[var].get("fscl", 1.0),
        }
        for var in variables
    }
    
    # --- run the calculator (unchanged) ---
    calc = BiasSigAtLevelsCalculator(
        regnam=regnam,
        tstart=tstart,
        tend=tend,
        path_in=path_template,
        out_path=out_path,
        model_list=list(exp_dict.keys()),
        ref_dict=REF_DICT,
        exp_dict=exp_dict,
        var_dict=VAR_DICT,
        force=False,
    )
    
    calc.compute(
        variables=variables,
        levels_hpa=levels_hpa,  # ignored for surface vars
        alpha=alpha,
        tol_hpa=tol_hpa,
    )

[LOAD] Model data: CLIM
[LOAD] Model data: UNet
[LOAD] Model data: UNetMP
[LOAD] Model data: IUNet
[LOAD] Model data: MnM
[INFO] Variable: U
[INFO]   Experiment: CLIM
[SKIP] /pscratch/sd/z/zhan391/SEACROGS_project/paper_material/method_paper/fig_data/horizontal_bias/U_global_biassigPL_CLIM_2012-2016.nc exists.
[INFO]   Experiment: UNet
[SKIP] /pscratch/sd/z/zhan391/SEACROGS_project/paper_material/method_paper/fig_data/horizontal_bias/U_global_biassigPL_UNet_2012-2016.nc exists.
[INFO]   Experiment: UNetMP
[SKIP] /pscratch/sd/z/zhan391/SEACROGS_project/paper_material/method_paper/fig_data/horizontal_bias/U_global_biassigPL_UNetMP_2012-2016.nc exists.
[INFO]   Experiment: IUNet
[SKIP] /pscratch/sd/z/zhan391/SEACROGS_project/paper_material/method_paper/fig_data/horizontal_bias/U_global_biassigPL_IUNet_2012-2016.nc exists.
[INFO]   Experiment: MnM
[SKIP] /pscratch/sd/z/zhan391/SEACROGS_project/paper_material/method_paper/fig_data/horizontal_bias/U_global_biassigPL_MnM_2012-2016.nc exists.
